# DAEN 328 Assignment 6, optional example: three services (a loader and a dashboard)

**Optional. Not graded, not submitted.** This notebook runs a small, complete version of the Part B setup on toy data (20 quotes), so you can see every piece working together before, or while, you build Part B with your own Assignment 5 code. If Part B is going fine, you can skip it.

It assumes you have finished Sections 1 to 5 of the main Guide (`Assignment06_Guide.ipynb`).

The quotes app from Section 5 has two weaknesses that would break a real pipeline:
1. the app may start before Postgres is ready (the "connection refused" race), and
2. it creates its table from Python code on the fly, instead of the database having its schema from the start.

This example fixes both and adds a one-shot loader and a Streamlit dashboard. The files are in **`Assignment06_Guide_Files/quotes_stack/`**. Run them as they are first, then read the explanation.

```
quotes_stack/
├── docker-compose.yml
├── .env.sample              copy to .env
├── db/init/01_schema.sql    creates the tables automatically
├── data/quotes.csv          20 rows, one of them bad on purpose
├── seed/                    the one-shot loader
│   ├── Dockerfile
│   ├── requirements.txt
│   └── seed.py
└── web/                     the Streamlit dashboard
    ├── Dockerfile
    ├── requirements.txt
    └── app.py
```

## 1. Run it

```bash
cd quotes_stack
cp .env.sample .env          # Windows PowerShell: copy .env.sample .env
docker compose up --build
```

Watch the log. You should see, in this order:
1. `db` starts and becomes **healthy**;
2. `seed` prints `Rows read: 20   good: 19   rejected: 1` and its counts, then **exits with code 0**;
3. `web` starts and prints `You can now view your Streamlit app in your browser`.

Open **http://localhost:8501**. Stop everything with **Ctrl+C**.

The flow:

```
 docker compose up --build
          │
          ▼
   ┌─────────────┐  healthy   ┌─────────────┐  exited 0   ┌─────────────┐
   │     db      │ ─────────► │    seed     │ ──────────► │     web     │
   │ postgres:16 │            │ runs once,  │             │  Streamlit  │
   │ runs schema │            │ loads CSV   │             │  port 8501  │
   └─────────────┘            └─────────────┘             └─────────────┘
          ▲  pgdata volume          │ reads ./data                │
          └─────────────────────────┴──── both connect to host "db"
```

## 2. What each piece does

**The database creates its own tables.** The `postgres` image runs every `.sql` file in `/docker-entrypoint-initdb.d/` when it starts **with an empty data volume**. The compose file mounts `./db/init` there:

```yaml
    volumes:
      - pgdata:/var/lib/postgresql/data
      - ./db/init:/docker-entrypoint-initdb.d:ro
```

So nobody clicks anything in pgAdmin: a brand new database already has its tables. In Part B, your Assignment 4 `EmployeeDB_schema.sql` goes there. (`:ro` means read-only.)

> **Important:** init scripts run **only once, on an empty volume.** If the volume already holds a database, Postgres skips them. Editing `01_schema.sql` later changes nothing until the volume is gone. Part C asks about exactly this.

**A healthcheck defines "ready".**

```yaml
    healthcheck:
      test: ["CMD-SHELL", "pg_isready -U $${POSTGRES_USER} -d $${POSTGRES_DB}"]
      interval: 3s
      timeout: 3s
      retries: 20
```

`pg_isready` asks Postgres whether it accepts connections. Compose runs it every 3 seconds, and the service only counts as **healthy** once it succeeds. (`$$` stops Compose from filling the variable in itself, so the shell inside the container does it.)

**`depends_on` with conditions sets the order.**

```yaml
  seed:
    depends_on:
      db:
        condition: service_healthy                  # wait for the healthcheck
  web:
    depends_on:
      seed:
        condition: service_completed_successfully   # wait for seed to exit with code 0
```

This is what plain `depends_on: - db` in Section 5 could not do. If `seed.py` crashes, it exits with code 1 and `web` never starts, so nobody sees a half-empty dashboard. That is why `seed.py` ends with:

```python
    except Exception as exc:
        print(f"SEED FAILED: {exc}", file=sys.stderr)
        sys.exit(1)
```

**Services find each other by name.** Inside the Compose network, the database's address is **`db`**, the service name. That is why the compose file passes `DB_HOST: db`. Inside a container, `localhost` means *that container itself*, and there is no Postgres inside the `seed` or `web` container.

**Settings come from environment variables, not from the code.**

```python
DB_PARAMS = {
    "host": os.environ["DB_HOST"],
    "port": os.environ.get("DB_PORT", "5432"),
    "dbname": os.environ["DB_NAME"],
    "user": os.environ["DB_USER"],
    "password": os.environ["DB_PASSWORD"],
}
```

The values travel `.env` → `docker-compose.yml` (`${DB_PASSWORD}`) → the container's environment → `os.environ` in Python. No password is written in any `.py` file, Dockerfile or YAML file.

**The data file is mounted, not copied.** `./data:/data:ro` makes the laptop's `data` folder appear inside the `seed` container at `/data`. The CSV is not baked into the image, so a new CSV does not need a rebuild, and a large file does not bloat the image.

**The loader is safe to run again.** It uses `ON CONFLICT ... DO NOTHING` (the patterns from the Assignment 5 Guide) and prints a before, after and added count for every table, so its log proves what happened.

**The dashboard listens on all addresses and publishes a port.**

```dockerfile
CMD ["streamlit", "run", "app.py", "--server.address=0.0.0.0", "--server.port=8501", "--server.headless=true"]
```

```yaml
    ports:
      - "8501:8501"          # laptop port : container port
```

`0.0.0.0` makes Streamlit accept connections from outside its container; without it, your browser cannot reach it. `ports` connects port 8501 on your laptop to port 8501 in the container. Notice `db` has **no** `ports` entry: only `seed` and `web` need it, and they reach it over the Compose network. This also avoids a clash with the PostgreSQL you installed for Assignment 4, which already uses port 5432 on your laptop.

## 3. Run it again, then start from nothing

With the stack stopped (Ctrl+C), run `docker compose up` again (no `--build` needed, nothing changed). Look at the `seed` log: every `added` is 0, because the data is still in the `pgdata` volume and the loader skips what is already there.

Then the full reset, the way a grader or a new teammate starts:

```bash
docker compose down -v
docker compose up --build
```

`down -v` deletes the volume, so Postgres starts empty, runs `01_schema.sql` again, and `seed` loads everything again. **If your stack survives this with no manual steps, it will run on anyone's machine.** That is the standard for Part B and for the term project.

Useful while it runs (in a second terminal, same folder):

```bash
docker compose ps -a                     # status of every service, including exited ones
docker compose logs seed                 # everything the loader printed
docker compose exec db psql -U postgres -d quotesdb -c "SELECT COUNT(*) FROM quotes;"
```

`up -d` starts the stack in the background (detached) if you want your terminal back; then use `logs` to read output and `down` to stop.

When you are done: `docker compose down -v`. Then go back to Section 6 of the main Guide and build the same thing with your own Assignment 5 code.